# Scale-matched control (no scripts, notebook only)

Is the weak-form drift a **blind spot**, or just the IC term being relatively **louder** in the weak loss?
Both forms use the *same* IC penalty, so the IC pull is identical; only the physics gradient it competes with differs.

**Run order:** M0 (load) → M1 (measure, minutes) → M2 (train the strong form at matched IC weights, ~20-40 min per run on a 20 GB slice) → M3 (evaluate + figure).
Every cell can be re-run: finished checkpoints are skipped, so a dropped session only costs the run in progress.
Do **not** run the experiment cells of `main.ipynb`; M0 loads only its definition cells.

In [ ]:
# %% [M0] Load main.ipynb's definition cells (sections 0-9 + new_net/get_model) into this kernel. Edit ROOT_DIR.
import os, json, copy, time, math

ROOT_DIR = "/content/amorflux"          # <- EDIT: the repo folder that holds data/, results/ and main.ipynb
NB_FILE  = os.path.join(ROOT_DIR, "main.ipynb")
CKPT_SUB = os.path.join("results", "model_files", "electrochem")   # where the cleaned repo keeps the checkpoints

os.environ["AMORFLUX_ROOT"] = ROOT_DIR
os.environ["AMORFLUX_OUT_DIR"] = ROOT_DIR
_cells = ["".join(c["source"]) for c in json.load(open(NB_FILE))["cells"] if c["cell_type"] == "code"]
_stop = next(i for i, c in enumerate(_cells) if "def train_strong(m_init" in c)
for _i, _src in enumerate(_cells[:_stop + 1]):
    if _i == 0:
        _src = _src.replace('drive.mount("/content/drive")', "pass")
    exec(compile(_src, f"main_cell{_i}", "exec"), globals())
    if _i == 0:
        RUN = {k: False for k in RUN}                       # skip solver checks
        # force every path to the repo, whatever cell 0 guessed (Colab vs local)
        ROOT = ROOT_DIR
        DATA_DIR = os.path.join(ROOT, "data")
        CKPT_DIR = os.path.join(ROOT, CKPT_SUB)
        RES_DIR = os.path.join(ROOT, "results")
        RESULTS_PATH = os.path.join(RES_DIR, "results.json")
        for _d in (DATA_DIR, CKPT_DIR, RES_DIR):
            os.makedirs(_d, exist_ok=True)

SEEDS3 = (42, 43, 44)
def _load(name):
    m = new_net()
    m.load_state_dict(torch.load(ckpt(name), map_location=dev, weights_only=False)["state_dict"])
    return m

_have = [s for s in SEEDS3 if os.path.exists(ckpt(f"warmstart_s{s}"))]
print(f"\nloaded {_stop + 1} definition cells | device {dev} | checkpoints in {CKPT_DIR}")
print(f"warm starts found for seeds {_have}" + ("" if _have else "  <-- none: check ROOT_DIR / CKPT_SUB"))

In [ ]:
# %% [M1] Measure the gradient balance at the warm start (no training, a few minutes)
# For each form: gradient of the IC term (weight 1), the physics terms, and the auxiliary terms, averaged over D fixed
# draws. The IC draws use the same seeds for both forms, so ||g_IC|| is identical; only the physics gradient differs.
D_DRAWS, B_ENVS = 10, 32

def _term_grads(m, form):
    b = [t[:B_ENVS] for t in dataset_tensors]
    params = list(m.parameters())
    flat = lambda gs: torch.cat([(x if x is not None else torch.zeros_like(p)).reshape(-1) for x, p in zip(gs, params)])
    acc = {}
    for d in range(D_DRAWS):
        torch.manual_seed(1000 + d)
        if form == "weak":
            rC, rP, bd, sm = compute_weak_loss_split(m, b, quad_tensors, TE, PHYSICS_PARAMS, N_t=5, device=dev)
            mass, charge = W_RES_C * rC, W_RES_PHI * rP
        else:
            rC, rP, bC, bP, bd, sm = compute_strong_loss_split(m, b, quad_tensors, PHYSICS_PARAMS, N_t=5, device=dev)
            mass, charge = W_RES_C * rC + bC, W_RES_PHI * rP + bP          # w_bc = 1, as in training
        torch.manual_seed(2000 + d)
        _, _, ic, eC, eP = compute_essential_penalties(m, b, device=dev)
        aux = W_BOUNDS * bd + W_SMOOTH * sm + W_ENDS_C * eC + W_ENDS_PHI * eP
        for k, L in dict(mass=mass, charge=charge, ic=ic, aux=aux).items():
            v = flat(torch.autograd.grad(L, params, retain_graph=True, allow_unused=True)).detach()
            acc[k] = acc.get(k, 0) + v / D_DRAWS
        m.zero_grad(set_to_none=True)
    acc["phys"] = acc["mass"] + acc["charge"]
    return acc

MEAS = {"per_seed": {}}
for s in _have:
    m = _load(f"warmstart_s{s}"); row = {}
    for form in ("weak", "strong"):
        G = _term_grads(m, form)
        nic = G["ic"].norm().item(); u = -G["ic"] / nic                  # unit direction of the IC pull
        row[form] = dict(ic=nic, phys=G["phys"].norm().item(), mass=G["mass"].norm().item(),
                         ratio=nic / G["phys"].norm().item(), ratio_massonly=nic / G["mass"].norm().item(),
                         cos=float(torch.dot(G["ic"], G["phys"]) / (nic * G["phys"].norm())),
                         resist_phys=float(torch.dot(G["phys"], u)), resist_aux=float(torch.dot(G["aux"], u)))
        flush()
    MEAS["per_seed"][s] = row
    print(f"\nseed {s}: ||g_IC|| = {row['weak']['ic']:.3e} (identical for both forms)")
    print(f"  {'':8}{'||g_phys||':>12}{'IC/phys':>10}{'cos(IC,phys)':>14}{'phys resists':>14}{'aux resists':>13}")
    for f in ("weak", "strong"):
        r = row[f]
        print(f"  {f:<8}{r['phys']:>12.3e}{r['ratio']:>10.3g}{r['cos']:>+14.3f}{r['resist_phys']:>+14.3e}{r['resist_aux']:>+13.3e}")
    del m

_rows = list(MEAS["per_seed"].values())
_k = [r["weak"]["ratio"] / r["strong"]["ratio"] for r in _rows]           # = ||g_phys_strong|| / ||g_phys_weak||
K_LOUD = math.exp(sum(map(math.log, _k)) / len(_k))
MEAS.update(k_per_seed=_k, k=K_LOUD,
            mean_ratio_weak=sum(r["weak"]["ratio"] for r in _rows) / len(_rows),
            mean_ratio_strong=sum(r["strong"]["ratio"] for r in _rows) / len(_rows))
json.dump(MEAS, open(os.path.join(RES_DIR, "matched_control_measure.json"), "w"), indent=2)

print(f"\nloudness ratio k = (IC/phys in weak) / (IC/phys in strong) = {K_LOUD:.3g}   "
      f"(per seed: {', '.join(f'{x:.3g}' for x in _k)})")
print("A strong-form IC weight of k makes the IC pull as loud, relative to physics, as in the weak form at w = 1.")
print("'phys resists' > 0 means a step along the IC pull raises that form's physics loss (it pushes back).")
if K_LOUD <= 1:
    print("\nk <= 1: the IC is already at least as loud in the strong form, and the strong form still holds.\n"
          "That alone argues against plain loss imbalance; M2 is then a confirmation.")
STRONG_W = sorted({float(f"{x:.2g}") for x in (K_LOUD / 3, K_LOUD, 3 * K_LOUD)})
print(f"\nM2 will train the strong form at IC weights {STRONG_W} (seed 42). Edit STRONG_W in M2 to override.")

In [ ]:
# %% [M2] Train the strong form from the warm start at the matched IC weights (in-kernel, one run at a time)
# Same recipe as run_jobs.py's worker: warm start, anchor off, production curriculum, only W_IC_C changed.
# Names match run_jobs.py: icw{w}_warm_strong_s{seed}. Finished runs are skipped on re-run.
STRONG_W = STRONG_W            # from M1; or set by hand, e.g. [10.0, 30.0, 100.0]
SEEDS_TRAIN = [42]             # add 43, 44 for the matched weight once seed 42 looks clear

for w in STRONG_W:
    for s in SEEDS_TRAIN:
        name = f"icw{w:g}_warm_strong_s{s}"
        torch.manual_seed(s)
        m0 = _load(f"warmstart_s{s}")
        with weights(W_ANCHOR=0.0, W_IC_C=float(w)):
            get_model(name, lambda: train_strong(m0, s), seed=s, init="warm", form="strong", band=0.0, w_ic=float(w))
        del m0; flush()
print("done:", [f"icw{w:g}_warm_strong_s{s}" for w in STRONG_W for s in SEEDS_TRAIN])

In [ ]:
# %% [M3] Evaluate weak and strong at every IC weight on one axis: IC loudness = w * ||g_IC|| / ||g_phys|| (warm start)
# CPU is fine. Writes results/matched_control_eval.json and results/paper_figs/fig5_matched.png/.pdf
import matplotlib.pyplot as plt
MEAS = json.load(open(os.path.join(RES_DIR, "matched_control_measure.json")))
_te = f32(np.load(os.path.join(DATA_DIR, "fdm_hold_te.npz"))["t_e"])
_REFS = {"0.25/Fo": (0.25 * _te, f32(np.load(os.path.join(DATA_DIR, "fdm_hold_te025.npz"))["C"])),
         "1/Fo": (_te, f32(np.load(os.path.join(DATA_DIR, "fdm_hold_te.npz"))["C"])),
         "t=1": (torch.ones_like(_te), fdm_C_ho)}

@torch.no_grad()
def _errs(m):
    m.eval(); hs = holdout_tensors[:3]; n = hs[0].shape[0]; out = {}
    for tag, (t, Ct) in _REFS.items():
        c = torch.cat([GRID_EVAL.unsqueeze(0).expand(n, -1, -1), t.view(n, 1, 1).expand(n, GRID_EVAL.shape[0], 1)], -1)
        C = m(*hs, c)[..., 0].view(n, len(fdm_x), len(fdm_y))
        out[tag] = ((C - Ct).flatten(1).norm(dim=1) / Ct.flatten(1).norm(dim=1)).mean().item()
    return out

_sw = sorted({float(f[3:].split("_warm_strong")[0]) for f in os.listdir(CKPT_DIR)
              if f.startswith("icw") and "_warm_strong_s" in f})
_ARMS = ([("weak", 0.0, "warm_weak_noIC_s{s}"), ("weak", 1.0, "warm_weak_s{s}")]
         + [("weak", w, f"icw{w:g}_warm_weak_s{{s}}") for w in (0.01, 0.03, 0.05, 0.1, 0.2, 0.3)]
         + [("strong", 0.0, "warm_strong_noIC_s{s}"), ("strong", 1.0, "warm_strong_s{s}")]
         + [("strong", w, f"icw{w:g}_warm_strong_s{{s}}") for w in _sw])

MATCHED = []
print(f"{'form':<8}{'w':>8}{'loudness':>10}{'n':>3}{'0.25/Fo':>10}{'1/Fo':>10}{'t=1':>10}   t=1 range")
for form, w, pat in _ARMS:
    have = [s for s in SEEDS3 if os.path.exists(ckpt(pat.format(s=s)))]
    if not have:
        continue
    per = []
    for s in have:
        m = _load(pat.format(s=s)); per.append(_errs(m)); del m
    flush()
    loud = w * (MEAS["mean_ratio_weak"] if form == "weak" else MEAS["mean_ratio_strong"])
    mean = {t: float(np.mean([p[t] for p in per])) for t in _REFS}
    MATCHED.append(dict(form=form, w=w, loudness=loud, n=len(have), mean=mean, per_seed=per))
    print(f"{form:<8}{w:>8g}{loud:>10.3g}{len(have):>3}" + "".join(f"{mean[t]:>10.3f}" for t in _REFS)
          + f"   [{min(p['t=1'] for p in per):.3f}-{max(p['t=1'] for p in per):.3f}]")
json.dump(dict(measure=MEAS, rows=MATCHED), open(os.path.join(RES_DIR, "matched_control_eval.json"), "w"), indent=2)
log_result("matched_control", dict(measure=MEAS, rows=MATCHED))

_col = dict(weak="#D55E00", strong="#009E73")
fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.6))
for ax, tag, title in zip(axes, ("0.25/Fo", "t=1"), ("early: $t=0.25/\\mathrm{Fo}$", "late: $t=1$")):
    for form in ("weak", "strong"):
        r = sorted([x for x in MATCHED if x["form"] == form and x["w"] > 0], key=lambda x: x["loudness"])
        if not r:
            continue
        ax.plot([x["loudness"] for x in r], [x["mean"][tag] for x in r], "-", color=_col[form], lw=1.2, label=f"{form} form")
        for x in r:
            ax.plot(x["loudness"], x["mean"][tag], "o", color=_col[form], ms=4, mfc=_col[form] if x["n"] >= 3 else "white")
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title(title)
    ax.set_xlabel("IC loudness at the warm start"); ax.set_ylabel("held-out rel. error")
axes[0].legend(frameon=False)
fig.text(0.99, -0.03, "open markers: 1 seed, filled: 3 seeds", ha="right", fontsize=7, color="0.4")
fig.tight_layout()
_od = os.path.join(RES_DIR, "paper_figs"); os.makedirs(_od, exist_ok=True)
for ext in ("png", "pdf"):
    fig.savefig(os.path.join(_od, f"fig5_matched.{ext}"), bbox_inches="tight", dpi=200)
plt.show()
print("At EQUAL loudness: strong well below weak -> weak-form blind spot. Curves overlap -> loss balance.")